Data obtained here: https://zenodo.org/records/14767363  
egrid details here: https://www.epa.gov/system/files/documents/2025-01/egrid2023_technical_guide.pdf  
ejscreen in action here: https://pedp-ejscreen.azurewebsites.net/  
ejscreen documentation here: https://www.epa.gov/system/files/documents/2024-07/ejscreen-tech-doc-version-2-3.pdf  
other resource i couldnt figure out: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/RLR5AX  
ejscreen tool archive: https://screening-tools.com/epa-ejscreen  


In [ ]:
# pip install pygris pandas


In [9]:
import pandas as pd
from pygris import block_groups
from geopy import distance
import numpy as np
from scipy.spatial import cKDTree


In [6]:
import requests
# ref https://www.geeksforgeeks.org/python/how-to-download-files-from-urls-with-python/#
url = 'https://www2.census.gov/geo/docs/reference/cenpop2020/blkgrp/CenPop2020_Mean_BG.txt'

response = requests.get(url)
file_path = 'data/CenPop2020_Mean_BG.txt'

if response.status_code == 200:
    with open(file_Path, 'wb') as file:
        file.write(response.content)
    print('File downloaded successfully')
else:
    print('Failed to download file')

File downloaded successfully


In [7]:
#block group means
mean_bg = pd.read_csv(file_path) #, dtype={'STATEFP': str, 'COUNTYFP': str, 'TRACTCE': str, 'BLKGRPCE': str, 'LATITUDE': str, 'LONGITUDE' : str} #STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,POPULATION,LATITUDE,LONGITUDE
    
mean_bg['STATEFP'] = mean_bg['STATEFP'].astype(str).str.zfill(2)
mean_bg['COUNTYFP'] = mean_bg['COUNTYFP'].astype(str).str.zfill(3)
mean_bg['TRACTCE'] = mean_bg['TRACTCE'].astype(str).str.zfill(6)
mean_bg['BLKGRPCE'] = mean_bg['BLKGRPCE'].astype(str).str.zfill(1)
mean_bg['ID'] = mean_bg['STATEFP']+ mean_bg['COUNTYFP'] + mean_bg['TRACTCE'] + mean_bg['BLKGRPCE']
mean_bg = mean_bg[['ID', 'LATITUDE', 'LONGITUDE']].set_index('ID')
mean_bg

,LATITUDE,LONGITUDE
ID,,
010010201001,32.464466,-86.486302
010010201002,32.482744,-86.486741
010010202001,32.478370,-86.474486
010010202002,32.466065,-86.470884
010010203001,32.477855,-86.459855
...,...,...
721537506011,18.019966,-66.840926
721537506012,18.020182,-66.844812
721537506013,18.015907,-66.847614


In [11]:
# get lats and longs of plants
egrid = pd.read_csv('data/egrid_counties.csv', index_col = 0)

#put plant loc points into radian tuples for kdt
plant_query = np.radians(np.array(list(zip(egrid['Plant latitude'], egrid['Plant longitude'])))) 
plant_kdt = cKDTree(plant_query) #for query


In [13]:
use_cols = [
    #'OBJECTID',
    'ID', 
    'REGION',
    'PEOPCOLOR', 
    'ACSTOTPOP', #Total population
    'LOWINCOME', 'ACSIPOVBAS', #Population for whom poverty status is determined
    'UNEMPLOYED',  'ACSUNEMPBAS', #Unemployment base--persons in civilian labor force (unemployment rate)
    'LINGISO', #Limited English speaking households
    'ACSTOTHH', #Households (for limited English speaking)
    'LESSHS', 
    'ACSEDUCBAS', #Population 25 years and over (use for less than hs)
    'UNDER5', 
    'OVER64',
    'P_LIFEEXPPCT', 
    'ST_ABBREV', 
    'CNTY_NAME'
]

cols_agg = {
    'ST_ABBREV': 'first',
    'REGION': 'first', #US EPA region number
    'CNTY_NAME': 'first',
    'P_LIFEEXPPCT': 'mean', #get an average percentile
    'PEOPCOLOR': 'sum',
    'ACSTOTPOP': 'sum',
    'LOWINCOME': 'sum',
    'ACSIPOVBAS': 'sum',
    'UNEMPLOYED': 'sum',
    'ACSUNEMPBAS': 'sum',
    'LINGISO': 'sum', 
    'ACSTOTHH': 'sum',
    'LESSHS': 'sum', 
    'ACSEDUCBAS': 'sum',
    'UNDER5': 'sum', 
    'OVER64': 'sum',
    
}

In [17]:

output = pd.DataFrame()
df = pd.read_csv('data/EJSCREEN_2023_BG_with_AS_CNMI_GU_VI.csv',  usecols = use_cols, encoding = 'latin1', chunksize = 10000)


for chunk in df:

    chunk['ID'] = chunk['ID'].astype(str).str.zfill(12)
    chunk = chunk.merge(mean_bg, how='left', left_on='ID', right_index=True)
    chunk['LATITUDE'] = pd.to_numeric(chunk['LATITUDE'], errors='coerce')
    chunk['LONGITUDE'] = pd.to_numeric(chunk['LONGITUDE'], errors='coerce')
    chunk = chunk.dropna(subset=['LATITUDE', 'LONGITUDE'])

    ## kdtree finding closest distance of block group to a plant
    bg_locs = np.radians(chunk[['LATITUDE', 'LONGITUDE']].to_numpy())
    
    #find nearest plant for each block group center, dd = distance in rad ref: 
    dd, ii = plant_kdt.query(bg_locs, k=1) 
    chunk['MIN_DISTANCE_MILES'] = dd * 3958.756  #convert rad to miles
    
    # non host communities must be > 3 miles and < 50 miles away from a plant
    chunk = chunk[chunk['MIN_DISTANCE_MILES'] > 3]]
    chunk = chunk[chunk['MIN_DISTANCE_MILES'] < 50 ]]
  
    if not chunk.empty:
        chunk['County FIPS'] = chunk['ID'].astype(str).str.zfill(12).str[:5] #fill front with 0s in case fips codes should be 12 digits
        chunk_grouped = chunk.groupby('County FIPS').agg(cols_agg) #group within chunk for efficiency, but will need to repeat
        output = pd.concat([chunk_grouped, output])
    
    #display(output)
    #break
output

,ST_ABBREV,REGION,CNTY_NAME,P_LIFEEXPPCT,PEOPCOLOR,ACSTOTPOP,LOWINCOME,ACSIPOVBAS,UNEMPLOYED,ACSUNEMPBAS,LINGISO,ACSTOTHH,LESSHS,ACSEDUCBAS,UNDER5,OVER64
County FIPS,,,,,,,,,,,,,,,,
53033,WA,10,King County,19.933014,126112,362175,34114,360873,7092,191913,4531,130882,8508,249238,21902,43924
53035,WA,10,Kitsap County,40.076471,67451,273072,52350,264476,5721,123728,1058,104977,9181,191555,15197,49202
53037,WA,10,Kittitas County,31.137931,7534,40748,11979,38244,1470,21819,179,17203,1838,24606,1715,6064
53039,WA,10,Klickitat County,18.571429,3312,16640,5347,16609,346,7387,55,6762,1349,12707,755,4106
53041,WA,10,Lewis County,54.859649,13310,74440,23827,73211,2368,32998,222,28492,5691,52843,4271,15243
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
05047,AR,6,Franklin County,75.125000,1077,11066,4606,10917,147,4774,0,4255,931,7458,491,2163
05049,AR,6,Fulton County,64.500000,802,12072,5888,11892,256,4337,3,4636,1224,8678,591,3090
05051,AR,6,Garland County,79.253968,15441,80988,31824,79786,2379,35521,435,34383,5891,58716,4184,18993


In [18]:
#final grouping bc same county could have been in different 'chunks'
final_df = output.groupby(output.index).agg(cols_agg).reset_index()

In [19]:
final_df

,County FIPS,ST_ABBREV,REGION,CNTY_NAME,P_LIFEEXPPCT,PEOPCOLOR,ACSTOTPOP,LOWINCOME,ACSIPOVBAS,UNEMPLOYED,ACSUNEMPBAS,LINGISO,ACSTOTHH,LESSHS,ACSEDUCBAS,UNDER5,OVER64
0,01001,AL,4,Autauga County,72.075000,15668,58239,17782,57790,752,26623,32,21856,4126,39614,3318,8815
1,01003,AL,4,Baldwin County,56.908257,39583,227131,57840,223772,3994,108361,730,87190,14555,161977,12035,46805
2,01005,AL,4,Barbour County,84.588235,13991,25259,11195,22250,808,9369,117,9088,4378,17995,1320,4801
3,01007,AL,4,Bibb County,86.500000,5816,22412,8485,21000,884,9107,23,7083,3125,16057,1196,3594
4,01009,AL,4,Blount County,72.976744,8289,58884,19518,58323,1554,25798,337,21300,6650,40668,3467,10584
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2534,56037,WY,8,Sweetwater County,43.636364,9109,41967,9231,41449,1485,22111,298,15340,1992,27431,2602,5304
2535,56039,WY,8,Teton County,15.250000,4681,23319,5002,23240,368,15320,466,9531,701,17659,1068,3609
2536,56041,WY,8,Uinta County,61.117647,2661,20514,5206,20267,348,10036,126,7675,851,13233,1413,3022
2537,56043,WY,8,Washakie County,31.750000,1405,6845,1591,6670,96,3593,10,2934,293,4710,348,1450


In [20]:
#renaming and calculating to match eGRID naming

# note that need to divide by different column values as specified in EJScreen technical guide
final_df['Total Population'] = final_df['ACSTOTPOP']
final_df['People of Color (%)'] = (final_df['PEOPCOLOR'] / final_df['ACSTOTPOP']) * 100
final_df['Low Income (%)'] = (final_df['LOWINCOME'] / final_df['ACSIPOVBAS']) * 100 #divide by Population for whom poverty status is determined
final_df['Unemployment Rate (%)'] = (final_df['UNEMPLOYED'] / final_df['ACSUNEMPBAS']) * 100 #Unemployment base--persons in civilian labor force (unemployment rate)
final_df['Limited English Speaking (%)'] = (final_df['LINGISO'] / final_df['ACSTOTHH']) * 100 #Households (for limited English speaking)
final_df['Less Than High School Education (%)'] = (final_df['LESSHS'] / final_df['ACSEDUCBAS']) * 100 #Population 25 years and over (use for less than hs)
final_df['Under Age 5 (%)'] = (final_df['UNDER5'] / final_df['ACSTOTPOP']) * 100
final_df['Over Age 64 (%)'] = (final_df['OVER64'] / final_df['ACSTOTPOP']) * 100
final_df['Plant state abbreviation'] = final_df['ST_ABBREV']
final_df['Plant county name'] = final_df['CNTY_NAME']
final_df['has_plant'] = 0
final_df['EPA Region'] = final_df['REGION']


In [21]:
df_drop = final_df.dropna()
save_df = df_drop[['County FIPS', 'Plant state abbreviation', 'Plant county name', 'EPA Region','has_plant','Total Population', 'People of Color (%)', 'Low Income (%)', 'Less Than High School Education (%)', 'Limited English Speaking (%)', 'Unemployment Rate (%)', 'Under Age 5 (%)', 'Over Age 64 (%)']]
save_df

,County FIPS,Plant state abbreviation,Plant county name,EPA Region,has_plant,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%),Under Age 5 (%),Over Age 64 (%)
0,01001,AL,Autauga County,4,0,58239,26.902934,30.770029,10.415510,0.146413,2.824625,5.697213,15.135905
1,01003,AL,Baldwin County,4,0,227131,17.427388,25.847738,8.985844,0.837252,3.685828,5.298704,20.607051
2,01005,AL,Barbour County,4,0,25259,55.390158,50.314607,24.328980,1.287412,8.624186,5.225860,19.007087
3,01007,AL,Bibb County,4,0,22412,25.950384,40.404762,19.461917,0.324721,9.706819,5.336427,16.036052
4,01009,AL,Blount County,4,0,58884,14.076829,33.465357,16.351923,1.582160,6.023723,5.887847,17.974322
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2534,56037,WY,Sweetwater County,8,0,41967,21.705149,22.270742,7.261857,1.942634,6.716114,6.200110,12.638502
2535,56039,WY,Teton County,8,0,23319,20.073760,21.523236,3.969647,4.889309,2.402089,4.579956,15.476650
2536,56041,WY,Uinta County,8,0,20514,12.971629,25.687078,6.430892,1.641694,3.467517,6.887979,14.731403
2537,56043,WY,Washakie County,8,0,6845,20.525931,23.853073,6.220807,0.340832,2.671862,5.084003,21.183346


In [22]:
save_df.to_csv('data/EJScreen_DEMO23.csv')